# ¿Cuánto vale una vivienda en California? Estimación del precio con Machine Learning clásico

### ACA 1 — Aprendizaje Automático
**Modelado con algoritmos clásicos de Machine Learning y despliegue interactivo en Streamlit**

Corporación Unificada Nacional de Educación Superior (CUN) · Escuela de Ingeniería · Especialización en Inteligencia Artificial
Docente: Dr. Hamilton Rivera Flor

| Integrantes |
|---|
| Santiago Toro Mesa |
| Darwin Lozano Lozano |

| Elemento | Definición |
|---|---|
| **Pilar** | Proyecto 3: **Ventas** — analítica comercial (mercado inmobiliario) |
| **Dataset** | California Housing Prices (Pace & Barry, 1997), versión de Kaggle — archivo `housing.csv` de la Clase 3 |
| **Variable objetivo** | `median_house_value`: valor de la vivienda típica de cada barrio, en dólares |
| **Tipo de problema** | Supervisado — **regresión** (el valor a estimar es un número continuo) |
| **App Streamlit** | ENLACE_APP |

# 1. El problema

## 1.1 Situación problema
Quien vende, compra o financia una vivienda necesita saber **cuánto vale**. Hoy ese valor se estima con avalúos que toman tiempo
y dependen del criterio de cada persona. Si existen datos de miles de barrios con su precio y sus características, un modelo
de Machine Learning puede **aprender la relación** entre esas características y el precio, y dar una estimación inmediata y objetiva.

## 1.2 Pregunta orientadora del proyecto
> **¿Se puede estimar el valor de venta de la vivienda típica de un barrio de California a partir de su ubicación, el ingreso de sus hogares
> y las características de sus viviendas, y poner esa estimación al alcance de cualquier usuario en una aplicación interactiva?**

## 1.3 Por qué este dataset y este pilar
- El dataset **California Housing** está entre los datasets propuestos por el docente para el ACA 1 (Clase 3, diapositiva 80), con
  `MedHouseVal` (valor de la vivienda) como variable objetivo y problema de **regresión supervisada**.
- Lo ubicamos en el **pilar de Ventas** porque el resultado es una herramienta de **analítica comercial**: estimar el precio de venta
  de un producto (la vivienda) para apoyar decisiones de compra, venta o fijación de precios.
- Da **continuidad** al trabajo hecho en la materia *Programación para Analítica de Datos e IA*. Allá se hizo el análisis exploratorio y una
  primera comparación de modelos con la versión de scikit-learn. Aquí usamos la versión de **Kaggle** que trabaja el docente,
  que trae **vacíos reales** y una **columna de texto**, y añadimos ajuste de hiperparámetros, agrupamiento (K-Means) y la app en Streamlit.

## 1.4 Ruta de trabajo (flujo metodológico del enunciado)

| Paso | Qué se hace en este notebook | Sección |
|---|---|---|
| 1. Exploración y EDA | Conocer las columnas, sus rangos, la ubicación y las relaciones con el precio | 2 |
| 2. Limpieza y datos | Vacíos, categorías, valores tope, valores imposibles y nuevas variables | 3 |
| 3. Modelado clásico | Regresión lineal (referencia), Random Forest y su ajuste, K-Means | 4 y 6 |
| 4. Evaluación | MAE, RMSE, R², validación cruzada, residuos e importancia de variables | 5 |
| 5. Despliegue Streamlit | Exportar la configuración del modelo y publicar la app | 7 |

# 2. Paso 1 — Exploración y análisis exploratorio (EDA)

## 2.1 Librerías
| Librería | Para qué la usamos |
|---|---|
| `pandas` | Leer y manipular la tabla de datos |
| `numpy` | Cálculos numéricos |
| `matplotlib` y `seaborn` | Gráficas |
| `scikit-learn` | Preparar los datos, entrenar y evaluar los modelos de Machine Learning |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
import json

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score

sb.set_theme(style="whitegrid")
print("Librerías listas.")

## 2.2 Cargar los datos
Se lee el archivo desde el repositorio del proyecto en GitHub, para que el notebook corra igual en cualquier computador o en Colab.
Si no hay conexión, se puede subir `housing.csv` a Colab y se usa la copia local.

In [ ]:
# Enlace al archivo housing.csv guardado en el repositorio del proyecto (GitHub).
# Es el mismo archivo que el profesor compartió en la carpeta DataSets de la Clase 3.
enlace = "https://raw.githubusercontent.com/santiagotoromesa-cell/aca1-aprendizaje-automatico/main/housing.csv"

try:
    datos = pd.read_csv(enlace)
    print("Datos cargados desde GitHub.")
except Exception:
    datos = pd.read_csv("housing.csv")   # alternativa: archivo subido a Colab
    print("Datos cargados desde el archivo local housing.csv.")

print("Filas:", datos.shape[0], "| Columnas:", datos.shape[1])
datos.head()

## 2.3 Qué significa cada columna
Cada fila es **un barrio** (técnicamente, un "grupo de manzanas" del censo de 1990 de EE. UU.), no una casa.

| Columna | Qué es | Unidad |
|---|---|---|
| `longitude`, `latitude` | Ubicación del barrio | grados |
| `housing_median_age` | Antigüedad de la vivienda típica | años |
| `total_rooms` | Total de cuartos en **todo** el barrio | cuartos |
| `total_bedrooms` | Total de dormitorios en **todo** el barrio | dormitorios |
| `population` | Personas que viven en el barrio | personas |
| `households` | Número de hogares del barrio | hogares |
| `median_income` | Ingreso del hogar típico | decenas de miles de USD al año (3,5 = USD 35.000) |
| `ocean_proximity` | Cercanía al mar (texto) | categoría |
| **`median_house_value`** | **Valor de la vivienda típica (lo que queremos estimar)** | **USD** |

In [ ]:
datos.info()

**Lectura:** hay 20.640 barrios y 10 columnas. Todas son numéricas salvo `ocean_proximity` (texto). `total_bedrooms` tiene
**20.433** datos en lugar de 20.640: le faltan **207** valores. Ambos puntos se tratan en el Paso 2.

In [ ]:
datos.describe().round(2).T

**Lectura:** los totales (`total_rooms`, `population`) tienen máximos muy lejanos de su valor típico (39.320 cuartos contra una mediana
de 2.127), señal de barrios atípicos. El valor de la vivienda va de USD 14.999 a **USD 500.001**, un número sospechosamente redondo.

In [ ]:
datos["ocean_proximity"].value_counts()

**Lectura:** la categoría `ISLAND` (isla) solo tiene **5 barrios**. Es muy poca información para que un modelo aprenda algo de ella.

In [ ]:
datos.hist(figsize=(15, 9), bins=40, edgecolor="black")
plt.suptitle("Distribución de cada variable numérica", fontsize=14)
plt.tight_layout()
plt.show()

**Lectura:** en los histogramas de `median_house_value`, `housing_median_age` y `median_income` aparece una **barra aislada en el extremo
derecho**. Eso indica que el censo **cortó** esos valores en un tope: todo lo que superaba el límite se registró con el límite.

In [ ]:
print("Barrios con valor exactamente en el tope (USD 500.001):", (datos["median_house_value"] == 500001).sum())
print("Barrios con antigüedad exactamente en el tope (52 años):", (datos["housing_median_age"] == 52).sum())
print("Barrios con ingreso en el tope (15 o más):", (datos["median_income"] >= 15).sum())

El caso que más importa es el del **precio**: 965 barrios dicen "USD 500.001", pero su valor real es *"USD 500.000 o más"*, sin saber cuánto.
Si el modelo aprende de ellos, aprende un precio que no es el verdadero. En la materia de Programación medimos ese efecto: el modelo
se equivocaba **más del doble** en esos barrios y siempre hacia abajo. Por eso aquí se toma la decisión de retirarlos (sección 3.3).

In [ ]:
plt.figure(figsize=(10, 8))
sb.scatterplot(x="longitude", y="latitude", data=datos, hue="median_house_value",
               size="population", sizes=(5, 120), palette="viridis", alpha=0.4)
plt.title("Valor de la vivienda por ubicación en California")
plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Valor / población")
plt.show()

**Lectura:** los barrios más caros (amarillo) se concentran sobre la **costa**, alrededor de **San Francisco** (arriba a la izquierda) y de
**Los Ángeles y San Diego** (abajo). Hacia el interior el valor baja. La ubicación pesa en el precio.

In [ ]:
plt.figure(figsize=(9, 5))
sb.boxplot(x="ocean_proximity", y="median_house_value", data=datos,
           order=["INLAND", "<1H OCEAN", "NEAR OCEAN", "NEAR BAY", "ISLAND"])
plt.title("Valor de la vivienda según cercanía al mar")
plt.xlabel("Cercanía al mar")
plt.ylabel("Valor de la vivienda (USD)")
plt.show()

**Lectura:** los barrios del interior (`INLAND`) valen claramente menos que los cercanos al mar o a la bahía. Esta columna de texto
aporta información útil, así que hay que convertirla a números para que el modelo la pueda usar (sección 4.3).

In [ ]:
correlacion = datos.drop(columns=["ocean_proximity"]).corr()
plt.figure(figsize=(10, 7))
sb.heatmap(correlacion, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("Correlación entre variables numéricas")
plt.show()

correlacion["median_house_value"].sort_values(ascending=False).round(3)

**Lectura:** la variable más relacionada con el precio es el **ingreso de los hogares** (correlación 0,69). Las columnas de totales
(`total_rooms`, `population`, `households`) casi no se relacionan con el precio, porque dependen del **tamaño del barrio** y no de
cómo es la vivienda. Por eso en el Paso 2 se convierten a valores **por hogar**.

### Resumen del Paso 1
1. Faltan 207 datos en `total_bedrooms`.
2. Hay una columna de texto (`ocean_proximity`) con una categoría de solo 5 barrios.
3. El precio está cortado en USD 500.001 en 965 barrios.
4. Hay barrios con valores extremos en los totales.
5. El ingreso y la ubicación son las pistas más fuertes sobre el precio.

# 3. Paso 2 — Limpieza y preparación de los datos

Cada decisión se toma por una razón concreta y se mide cuántos barrios afecta.

## 3.1 Vacíos y duplicados

In [ ]:
print("CASILLAS VACÍAS POR COLUMNA")
print(datos.isna().sum())
print()
print("Filas repetidas:", datos.duplicated().sum())

- **Duplicados:** no hay filas repetidas.
- **Vacíos:** 207 en `total_bedrooms` (1 % de los barrios). Las opciones vistas en clase son **eliminar** las filas (`dropna`) o **rellenarlas**
  (`fillna`). Se decide **rellenar con la mediana**, por tres razones:
  1. Son pocos datos y el resto de la fila es válido; eliminarlos desperdicia información.
  2. La **mediana** no se deja arrastrar por los barrios extremos, a diferencia del promedio (la media de esta columna es 538 y la mediana 435).
  3. El relleno se hace **dentro del modelo** y se calcula **solo con los datos de entrenamiento**. Así los datos de prueba no "filtran" información
     al modelo (fuga de información o *data leakage*).

## 3.2 Barrios de isla

In [ ]:
datos_limpios = datos.copy()
filas_inicio = len(datos_limpios)

# Paso 2.2: retirar los 5 barrios de isla (muy pocos para que el modelo aprenda de ellos)
datos_limpios = datos_limpios[datos_limpios["ocean_proximity"] != "ISLAND"]
print("Barrios retirados por ser de isla:", filas_inicio - len(datos_limpios))

## 3.3 Barrios con precio cortado en el tope

In [ ]:
# Paso 2.3: retirar los barrios cuyo precio quedó cortado en el tope del censo
antes = len(datos_limpios)
datos_limpios = datos_limpios[datos_limpios["median_house_value"] < 500001]
print("Barrios retirados por precio cortado en USD 500.001:", antes - len(datos_limpios))

Se retiran porque su valor real **se desconoce**. Consecuencia que se debe comunicar: el modelo **solo es válido para viviendas de hasta
USD 500.000**, y la app lo advierte.

## 3.4 Nuevas variables "por hogar"
Un barrio con 6.000 cuartos no tiene viviendas más grandes que uno con 1.000; puede que simplemente tenga más casas. Por eso se crean:

| Nueva variable | Cálculo | Qué describe |
|---|---|---|
| `cuartos_por_hogar` | cuartos ÷ hogares | Tamaño de la vivienda típica |
| `personas_por_hogar` | personas ÷ hogares | Ocupación de la vivienda típica |
| `dormitorios_por_cuarto` | dormitorios ÷ cuartos | Qué parte de la vivienda son dormitorios |

In [ ]:
# Paso 2.4: crear variables "por hogar", que sí describen la vivienda típica del barrio
datos_limpios["cuartos_por_hogar"] = datos_limpios["total_rooms"] / datos_limpios["households"]
datos_limpios["personas_por_hogar"] = datos_limpios["population"] / datos_limpios["households"]
datos_limpios["dormitorios_por_cuarto"] = datos_limpios["total_bedrooms"] / datos_limpios["total_rooms"]

datos_limpios[["cuartos_por_hogar", "personas_por_hogar", "dormitorios_por_cuarto"]].describe().round(2).T

## 3.5 Barrios con valores imposibles
Las nuevas variables revelan barrios que no son zonas residenciales normales: hasta **132 cuartos por vivienda** o **1.243 personas por
vivienda** (probablemente hoteles, cárceles o residencias universitarias). Se aplican las mismas dos reglas de sentido común del trabajo anterior:

In [ ]:
# Paso 2.5: retirar barrios con valores imposibles para una zona residencial
regla_1 = datos_limpios["cuartos_por_hogar"] > 20     # más de 20 cuartos por vivienda
regla_2 = datos_limpios["personas_por_hogar"] > 10    # más de 10 personas por vivienda

antes = len(datos_limpios)
datos_limpios = datos_limpios[~(regla_1 | regla_2)]
print("Barrios retirados por valores imposibles:", antes - len(datos_limpios))

## 3.6 Distancia a la ciudad grande más cercana
El mapa mostró que los barrios caros están cerca de San Francisco y Los Ángeles. Se calcula la distancia en kilómetros a la más cercana de las dos,
con una aproximación sencilla (1 grado ≈ 111 km).

In [ ]:
# Paso 2.6: distancia en km a la ciudad grande más cercana (Los Ángeles o San Francisco)
def distancia_km(lat, lon, lat_ciudad, lon_ciudad):
    lado_vertical = (lat - lat_ciudad) * 111
    lado_horizontal = (lon - lon_ciudad) * 111 * np.cos(np.radians(lat))
    return np.sqrt(lado_vertical ** 2 + lado_horizontal ** 2)

km_LA = distancia_km(datos_limpios["latitude"], datos_limpios["longitude"], 34.05, -118.24)
km_SF = distancia_km(datos_limpios["latitude"], datos_limpios["longitude"], 37.77, -122.42)
datos_limpios["km_a_ciudad"] = np.minimum(km_LA, km_SF)

print("Correlación de la distancia con el valor:",
      round(datos_limpios["km_a_ciudad"].corr(datos_limpios["median_house_value"]), 3))

La distancia tiene correlación **−0,45** con el precio: a más kilómetros, menor valor. Es la segunda variable más relacionada con el precio,
después del ingreso.

## 3.7 Resumen de la limpieza

In [ ]:
print("RESUMEN DE LA LIMPIEZA")
print("Barrios al inicio       :", filas_inicio)
print("Barrios para el modelo  :", len(datos_limpios))
print("Barrios retirados       :", filas_inicio - len(datos_limpios),
      "(", round((filas_inicio - len(datos_limpios)) / filas_inicio * 100, 2), "% )")
print("Vacíos que quedan en total_bedrooms y dormitorios_por_cuarto:",
      datos_limpios["dormitorios_por_cuarto"].isna().sum(), "-> se imputan dentro del modelo")

| Decisión | Barrios afectados | Tratamiento |
|---|---|---|
| Vacíos en `total_bedrooms` | 207 | Se rellenan con la mediana del entrenamiento, dentro del modelo |
| Duplicados | 0 | No aplica |
| Categoría `ISLAND` | 5 | Se retiran (muy pocos para aprender) |
| Precio cortado en USD 500.001 | 965 | Se retiran (valor real desconocido) |
| Valores imposibles | 100 | Se retiran (no son zonas residenciales) |
| Totales del barrio | — | Se reemplazan por variables por hogar |
| `ocean_proximity` (texto) | — | Se convierte en columnas 0/1 (sección 4.3) |
| **Total** | **1.070 retirados (5,2 %)** | **Quedan 19.570 barrios** |

# 4. Paso 3 — Modelado clásico

## 4.1 Variables de entrada (X) y variable objetivo (y)
Se usan 9 variables que **cualquier persona podría conocer o consultar** de un barrio, porque son las mismas que pedirá la app.

In [ ]:
variables_numericas = ["longitude", "latitude", "housing_median_age", "median_income",
                       "cuartos_por_hogar", "personas_por_hogar", "dormitorios_por_cuarto", "km_a_ciudad"]
variables_texto = ["ocean_proximity"]

X = datos_limpios[variables_numericas + variables_texto]
y = datos_limpios["median_house_value"]

print("Variables predictoras (X):", X.shape[1])
print("Variable objetivo (y)   :", y.name)

## 4.2 Separar entrenamiento y prueba
- **Entrenamiento (80 %):** barrios con los que el modelo aprende.
- **Prueba (20 %):** barrios que el modelo **nunca ve** mientras aprende; se usan al final para calificarlo, como un examen.
- `random_state=42` fija el azar: quien corra el notebook obtiene la misma partición y los mismos resultados (**reproducibilidad**).

In [ ]:
X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.20, random_state=42)

print("Barrios para entrenar:", len(X_ent))
print("Barrios para probar  :", len(X_pru))
print("Valor promedio en entrenamiento: USD", round(y_ent.mean()))
print("Valor promedio en prueba       : USD", round(y_pru.mean()))

Los promedios de los dos grupos son casi iguales, así que el grupo de prueba representa bien al total.

## 4.3 Preparación automática de los datos (pipeline)
Para que el relleno de vacíos, el escalado y la conversión del texto se aprendan **solo del entrenamiento**, se arma una "línea de
preparación" (*pipeline*) que se ejecuta en orden cada vez que el modelo entrena o estima:

| Paso | Regresión lineal | Random Forest | Por qué |
|---|---|---|---|
| Rellenar vacíos con la mediana | Sí | Sí | Ningún modelo acepta casillas vacías |
| Escalar (`StandardScaler`) | **Sí** | No | La regresión lineal es sensible a las unidades (ingreso de 0,5 a 15 contra distancia de 0 a 900 km). Los árboles solo hacen cortes del tipo "¿ingreso > 5?" y no les afecta la escala |
| Convertir texto a 0/1 (`OneHotEncoder`) | Sí | Sí | Equivale a los *dummies* (`pd.get_dummies`) vistos en clase: una columna por categoría |

In [ ]:
# Preparación para la regresión lineal: imputar con la mediana, escalar y convertir texto a columnas 0/1
preparacion_lineal = ColumnTransformer([
    ("numericas", Pipeline([("imputar", SimpleImputer(strategy="median")),
                            ("escalar", StandardScaler())]), variables_numericas),
    ("texto", OneHotEncoder(handle_unknown="ignore"), variables_texto)
])

# Preparación para el bosque aleatorio: igual, pero sin escalar (los árboles no lo necesitan)
preparacion_bosque = ColumnTransformer([
    ("numericas", SimpleImputer(strategy="median"), variables_numericas),
    ("texto", OneHotEncoder(handle_unknown="ignore"), variables_texto)
])
print("Preparaciones definidas.")

## 4.4 La línea base: responder siempre el promedio
Antes de entrenar, se mide qué tan mal le iría a alguien que, sin modelo, respondiera siempre el precio promedio. **Todo modelo tiene que
superar esta cifra** para justificar su uso.

In [ ]:
def calificar(nombre, y_real, y_estimado):
    return {"Modelo": nombre,
            "MAE (USD)": round(mean_absolute_error(y_real, y_estimado)),
            "RMSE (USD)": round(np.sqrt(mean_squared_error(y_real, y_estimado))),
            "R2": round(r2_score(y_real, y_estimado), 4)}

estimado_promedio = np.full(len(y_pru), y_ent.mean())
calificar("Responder siempre el promedio", y_pru, estimado_promedio)

## 4.5 Modelo 1 — Regresión lineal (referencia)
Supone que el precio es una **suma ponderada** de las variables: tanto más por cada punto de ingreso, tanto menos por cada kilómetro.
Es el modelo visto en clase con este mismo dataset y es el más fácil de explicar. Sirve como **referencia**: un modelo más complejo solo se justifica si le gana.

In [ ]:
modelo_lineal = Pipeline([("preparacion", preparacion_lineal), ("modelo", LinearRegression())])
modelo_lineal.fit(X_ent, y_ent)

estimado_lineal = modelo_lineal.predict(X_pru)
calificar("Regresión lineal", y_pru, estimado_lineal)

## 4.6 Modelo 2 — Random Forest (bosque aleatorio)
**Por qué este algoritmo:**
- Un **árbol de decisión** (visto en clase) responde con preguntas encadenadas: *¿el ingreso es mayor que 5? ¿está a menos de 30 km de la ciudad?*
  Capta relaciones que una línea recta no puede (por ejemplo, que el precio sube mucho en la costa y poco en el interior), pero tiende a **memorizar**.
- Un **Random Forest** entrena **muchos árboles**, cada uno con una muestra distinta de barrios y de variables, y **promedia** sus respuestas.
  Los errores de cada árbol se compensan entre sí y el resultado es más preciso y estable.
- En la materia de Programación fue el que mejor estimó el precio entre tres modelos. Aquí se da el paso siguiente: **ajustar su configuración**.

Primero se entrena con la configuración por defecto de scikit-learn:

In [ ]:
bosque_base = Pipeline([("preparacion", preparacion_bosque),
                        ("modelo", RandomForestRegressor(random_state=42, n_jobs=-1))])
bosque_base.fit(X_ent, y_ent)

estimado_bosque_base = bosque_base.predict(X_pru)
calificar("Random Forest (configuración por defecto)", y_pru, estimado_bosque_base)

## 4.7 Ajuste de hiperparámetros
Los **hiperparámetros** son las "perillas" del modelo que no se aprenden de los datos, sino que se fijan antes de entrenar.
`RandomizedSearchCV` prueba **10 combinaciones al azar** y califica cada una con **validación cruzada de 3 partes**
(entrena con 2/3 del entrenamiento y prueba con el 1/3 restante, tres veces). Se elige la combinación con **menor error promedio**.
Los datos de prueba no se tocan en esta búsqueda.

In [ ]:
opciones = {
    "modelo__n_estimators": [100, 200],          # cuántos árboles
    "modelo__max_depth": [None, 15, 25],         # cuántas preguntas seguidas puede hacer cada árbol
    "modelo__min_samples_leaf": [1, 2, 4],       # mínimo de barrios en cada respuesta final
    "modelo__max_features": [1.0, 0.5, "sqrt"]   # cuántas variables mira cada árbol en cada pregunta
}

busqueda = RandomizedSearchCV(
    bosque_base, opciones, n_iter=10, cv=3,
    scoring="neg_mean_absolute_error", random_state=42, n_jobs=-1
)
busqueda.fit(X_ent, y_ent)   # tarda entre 2 y 5 minutos en Colab

tabla_busqueda = pd.DataFrame(busqueda.cv_results_)[["params", "mean_test_score", "mean_fit_time"]]
tabla_busqueda["Error promedio (USD)"] = (-tabla_busqueda["mean_test_score"]).round(0)
tabla_busqueda["Segundos por entrenamiento"] = tabla_busqueda["mean_fit_time"].round(1)
tabla_busqueda = tabla_busqueda.sort_values("Error promedio (USD)")
print("Mejor combinación:", busqueda.best_params_)
tabla_busqueda[["params", "Error promedio (USD)", "Segundos por entrenamiento"]]

**Lectura y decisión:**
- La mejor combinación usa **árboles de hasta 25 niveles**, **un barrio por hoja** y que cada árbol mire **solo una parte de las variables**
  en cada pregunta (`sqrt`). Esto último hace los árboles más distintos entre sí, y el promedio funciona mejor.
- La mejor combinación tiene 200 árboles, pero la misma combinación con **100 árboles** queda prácticamente igual (diferencia de unos
  USD 90 en el error promedio) y tarda **menos de la mitad** en entrenar. Para la app esto importa: el modelo se carga más rápido. Se elige
  **100 árboles**.

In [ ]:
parametros_finales = {k.replace("modelo__", ""): v for k, v in busqueda.best_params_.items()}
parametros_finales["n_estimators"] = 100   # ver explicación arriba: mismo error, la mitad de peso

modelo_final = Pipeline([("preparacion", preparacion_bosque),
                         ("modelo", RandomForestRegressor(**parametros_finales, random_state=42, n_jobs=-1))])
modelo_final.fit(X_ent, y_ent)

estimado_final = modelo_final.predict(X_pru)
print("Parámetros del modelo final:", parametros_finales)
calificar("Random Forest ajustado (modelo final)", y_pru, estimado_final)

# 5. Paso 4 — Evaluación del modelo

## 5.1 Las métricas, en el contexto de este problema

| Métrica | Qué mide | Cómo se lee aquí |
|---|---|---|
| **MAE** (error absoluto medio) | El error promedio, en dólares | "En promedio, el modelo se equivoca por tantos dólares en el valor de un barrio" |
| **RMSE** (raíz del error cuadrático medio) | Error promedio que castiga más los errores grandes | Si es bastante mayor que el MAE, hay **pocos barrios con errores grandes** |
| **R²** (coeficiente de determinación) | Qué parte de las diferencias de precio entre barrios explica el modelo | 0 = no explica nada (igual que el promedio); 1 = explica todo |
| **R² de validación cruzada** | El R² repetido en 5 particiones distintas del entrenamiento | Si varía poco (±), el modelo es **estable** |
| **R² de entrenamiento** | El R² sobre los datos que el modelo ya vio | Si es mucho mayor que el de prueba, hay **sobreajuste** (memorización) |

## 5.2 Comparación de los modelos

In [ ]:
comparacion = pd.DataFrame([
    calificar("Responder siempre el promedio", y_pru, estimado_promedio),
    calificar("Regresión lineal", y_pru, estimado_lineal),
    calificar("Random Forest por defecto", y_pru, estimado_bosque_base),
    calificar("Random Forest ajustado", y_pru, estimado_final),
])

# Validación cruzada (5 particiones) sobre el entrenamiento: qué tan estable es cada modelo
for nombre, modelo in [("Regresión lineal", modelo_lineal), ("Random Forest ajustado", modelo_final)]:
    puntajes = cross_val_score(modelo, X_ent, y_ent, cv=5, scoring="r2", n_jobs=-1)
    comparacion.loc[comparacion["Modelo"] == nombre, "R2 validación cruzada"] = \
        f"{puntajes.mean():.4f} ± {puntajes.std():.4f}"

comparacion["R2 entrenamiento"] = [np.nan,
                                   round(r2_score(y_ent, modelo_lineal.predict(X_ent)), 4),
                                   round(r2_score(y_ent, bosque_base.predict(X_ent)), 4),
                                   round(r2_score(y_ent, modelo_final.predict(X_ent)), 4)]
comparacion

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.5))
nombres = ["Promedio", "Lineal", "RF defecto", "RF ajustado"]
ejes[0].bar(nombres, comparacion["MAE (USD)"], color="#2b6cb0")
ejes[0].set_title("Error promedio (MAE): entre más bajo, mejor")
ejes[0].set_ylabel("USD")
ejes[1].bar(nombres, comparacion["R2"], color="#2f855a")
ejes[1].set_title("R²: entre más cerca de 1, mejor")
ejes[1].set_ylim(0, 1)
for eje in ejes:
    for barra in eje.patches:
        eje.annotate(f"{barra.get_height():,.2f}" if barra.get_height() < 2 else f"{barra.get_height():,.0f}",
                     (barra.get_x() + barra.get_width() / 2, barra.get_height()), ha="center", va="bottom")
plt.tight_layout()
plt.show()

**Lectura de los resultados:**
1. **Los dos modelos superan con amplitud la línea base.** Responder el promedio equivoca en unos USD 79.300 por barrio; la regresión lineal baja
   el error a USD 42.200 y el Random Forest ajustado a **USD 27.000**.
2. **El Random Forest ajustado es el mejor:** explica el **82 %** de las diferencias de precio entre barrios (R² = 0,82), contra 66 % de la
   regresión lineal. El ajuste de hiperparámetros redujo el error del Random Forest en unos **USD 1.200 por barrio (4 %)** frente a la
   configuración por defecto.
3. **Es estable:** en validación cruzada su R² es 0,81 con una variación de apenas ±0,006 entre particiones.
4. **Sobreajuste, explicado:** el R² de entrenamiento (0,97) es mayor que el de prueba (0,82). En un Random Forest esto es normal, porque cada
   árbol llega hasta casi un barrio por hoja. Lo importante es que el desempeño en barrios **nunca vistos** (prueba y validación cruzada) es alto
   y estable; esa es la cifra que se reporta.
5. **RMSE mayor que MAE** (41.900 contra 27.000): la mayoría de los barrios se estima bien y **unos pocos** se estiman con errores grandes.
   En la sección 5.4 se ve cuáles son.

## 5.3 Valores reales contra valores estimados
Cada punto es un barrio de prueba. La **línea roja** es el acierto perfecto: entre más cerca estén los puntos, mejor.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 6))
for eje, (nombre, estimado) in zip(ejes, [("Regresión lineal", estimado_lineal),
                                          ("Random Forest ajustado", estimado_final)]):
    eje.scatter(y_pru, estimado, s=5, alpha=0.2, color="#2b6cb0")
    eje.plot([0, 520000], [0, 520000], color="red", linestyle="--", label="Acierto perfecto")
    eje.set_xlim(0, 520000)
    eje.set_ylim(-50000, 600000)
    eje.set_title(f"{nombre}\nR² = {r2_score(y_pru, estimado):.3f}")
    eje.set_xlabel("Valor real (USD)")
    eje.set_ylabel("Valor estimado (USD)")
    eje.legend()
plt.tight_layout()
plt.show()

**Lectura:** la nube del Random Forest es mucho más angosta y pegada a la línea roja. La regresión lineal se abre y llega a estimar
valores **negativos** en algunos barrios, algo imposible para un precio: la línea recta no logra seguir la forma real de los datos.

## 5.4 Análisis de residuos del modelo final
El **residuo** es la diferencia entre el valor real y el estimado. Si el modelo es justo, los residuos deben quedar repartidos alrededor de cero.

In [ ]:
residuos = y_pru - estimado_final

fig, ejes = plt.subplots(1, 2, figsize=(14, 5))
ejes[0].scatter(estimado_final, residuos, s=5, alpha=0.2, color="#2b6cb0")
ejes[0].axhline(0, color="red", linestyle="--")
ejes[0].set_title("Residuos contra valor estimado")
ejes[0].set_xlabel("Valor estimado (USD)")
ejes[0].set_ylabel("Residuo = real - estimado (USD)")
ejes[1].hist(residuos, bins=60, color="#2b6cb0", edgecolor="white")
ejes[1].axvline(0, color="red", linestyle="--")
ejes[1].set_title("Cómo se reparten los residuos")
ejes[1].set_xlabel("Residuo (USD)")
plt.tight_layout()
plt.show()

print("Residuo promedio: USD", round(residuos.mean()))
print("Barrios con error menor a USD 25.000:", round((residuos.abs() < 25000).mean() * 100, 1), "%")
print("Barrios con error menor a USD 50.000:", round((residuos.abs() < 50000).mean() * 100, 1), "%")

**Lectura:**
- El histograma es una campana centrada en cero y el residuo promedio es cercano a cero (menos de USD 1.000): el modelo **no infla ni desinfla**
  los precios de forma sistemática.
- Cerca del **65 %** de los barrios se estima con menos de USD 25.000 de error, y cerca del **86 %** con menos de USD 50.000.

In [ ]:
rangos = pd.cut(y_pru, bins=[0, 100000, 200000, 300000, 400000, 500001],
                labels=["< 100 mil", "100-200 mil", "200-300 mil", "300-400 mil", "400-500 mil"])
error_por_rango = pd.DataFrame({"rango": rangos, "error": residuos.abs(), "real": y_pru})
error_por_rango = error_por_rango.groupby("rango", observed=True).agg(
    barrios=("error", "size"),
    error_promedio_USD=("error", "mean"),
    valor_promedio_USD=("real", "mean"))
error_por_rango["error_%_del_valor"] = (error_por_rango["error_promedio_USD"] / error_por_rango["valor_promedio_USD"] * 100).round(1)
error_por_rango.round(0)

**Lectura:** en porcentaje, el error es menor en los barrios de **USD 100.000 a 400.000** (11 % a 14 % del valor). Es mayor en los extremos:
en los más baratos (24 %) y en los de **USD 400.000 a 500.000** (22 %), donde el modelo tiende a quedarse corto porque hay pocos barrios de
ese rango para aprender. Esto se informa en la app como limitación.

## 5.5 ¿En qué se apoya el modelo?

In [ ]:
nombres_columnas = modelo_final.named_steps["preparacion"].get_feature_names_out()
nombres_columnas = [n.split("__")[1] for n in nombres_columnas]
importancia = pd.Series(modelo_final.named_steps["modelo"].feature_importances_,
                        index=nombres_columnas).sort_values()

plt.figure(figsize=(9, 6))
plt.barh(importancia.index, importancia.values * 100, color="#2b6cb0")
plt.xlabel("Importancia (%)")
plt.title("¿En qué variables se apoya más el Random Forest?")
plt.show()

(importancia.sort_values(ascending=False) * 100).round(1)

**Lectura:** el **ingreso de los hogares** es la variable más importante, seguida de la **distancia a la ciudad** y de si el barrio está en el
**interior** (`INLAND`). Las tres juntas suman cerca de la mitad de la importancia. Esto coincide con lo que mostró el EDA: el precio depende
más de **dónde** está la vivienda y de **quién** vive en el barrio que del tamaño de la vivienda. La variable de distancia creada en la
limpieza resultó ser la segunda más útil: **la preparación de los datos mejoró el modelo**.

# 6. Complemento no supervisado — Zonas con K-Means

Como se hizo en clase con este mismo dataset, se usa **K-Means** para **agrupar barrios parecidos sin decirle al algoritmo el precio**.
Se agrupan por **ubicación e ingreso**. A diferencia del ejemplo de clase, primero se **escalan** las tres variables, porque la latitud y la
longitud varían en pocos grados y el ingreso en varias unidades; sin escalar, una variable dominaría a las otras.

## 6.1 ¿Cuántas zonas?

In [ ]:
variables_zona = ["latitude", "longitude", "median_income"]
escalador_zonas = StandardScaler()
Z = escalador_zonas.fit_transform(datos_limpios[variables_zona])

inercias, siluetas = [], []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(Z, km.labels_, sample_size=5000, random_state=42))

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].plot(range(2, 9), inercias, marker="o")
ejes[0].set_title("Método del codo")
ejes[0].set_xlabel("Número de zonas (k)")
ejes[0].set_ylabel("Inercia (dispersión dentro de las zonas)")
ejes[1].plot(range(2, 9), siluetas, marker="o", color="#2f855a")
ejes[1].set_title("Coeficiente de silueta")
ejes[1].set_xlabel("Número de zonas (k)")
plt.tight_layout()
plt.show()

**Lectura:** la inercia baja rápido hasta **k = 4** y después se aplana (el "codo"). Con 4 zonas la silueta todavía es aceptable (0,42).
Se eligen **4 zonas**, el mismo número usado en el ejemplo de clase.

In [ ]:
modelo_zonas = KMeans(n_clusters=4, n_init=10, random_state=42)
datos_limpios["zona"] = modelo_zonas.fit_predict(Z)

resumen_zonas = datos_limpios.groupby("zona").agg(
    barrios=("median_house_value", "size"),
    ingreso_medio=("median_income", "mean"),
    valor_medio_USD=("median_house_value", "mean"),
    latitud_media=("latitude", "mean"),
    km_a_ciudad=("km_a_ciudad", "mean")).round(2)

# Nombre sencillo para cada zona: norte o sur, e ingreso alto o medio-bajo
def nombrar_zona(fila):
    region = "Norte (San Francisco)" if fila["latitud_media"] > 36 else "Sur (Los Ángeles)"
    nivel = "ingreso alto" if fila["ingreso_medio"] > 4 else "ingreso medio-bajo"
    return region + " – " + nivel

resumen_zonas["nombre_zona"] = resumen_zonas.apply(nombrar_zona, axis=1)
datos_limpios["nombre_zona"] = datos_limpios["zona"].map(resumen_zonas["nombre_zona"])
resumen_zonas

In [ ]:
plt.figure(figsize=(9, 8))
sb.scatterplot(x="longitude", y="latitude", hue="nombre_zona", data=datos_limpios,
               palette="bright", s=8, alpha=0.6)
plt.title("Zonas encontradas por K-Means (ubicación + ingreso)")
plt.legend(title="Zona", loc="upper right")
plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.show()

**Lectura:** K-Means separó solo, sin conocer el precio, dos regiones (norte y sur) y dentro de cada una dos niveles de ingreso. Las zonas de
**ingreso alto** tienen un valor promedio de vivienda cercano a USD 260.000; las de ingreso medio-bajo, entre USD 140.000 y 165.000.
Es una confirmación independiente de que ubicación e ingreso explican buena parte del precio. La app muestra a qué zona pertenece
el barrio que el usuario ingresa.

## 6.2 Prueba con un barrio nuevo
Así se usa el modelo final con un barrio que no está en los datos (cerca de Los Ángeles, ingreso de USD 50.000 al año):

In [ ]:
barrio_nuevo = pd.DataFrame([{
    "longitude": -118.30, "latitude": 34.10, "housing_median_age": 30,
    "median_income": 5.0, "cuartos_por_hogar": 5.5, "personas_por_hogar": 2.8,
    "dormitorios_por_cuarto": 0.20,
    "km_a_ciudad": distancia_km(34.10, -118.30, 34.05, -118.24),
    "ocean_proximity": "<1H OCEAN"}])

print("Valor estimado de la vivienda típica de ese barrio: USD",
      f"{modelo_final.predict(barrio_nuevo)[0]:,.0f}")

# 7. Paso 5 — Despliegue en Streamlit

La app se construyó con **Streamlit** (Clases 4 y 5) y se publicó en **Streamlit Community Cloud** desde el repositorio de GitHub del proyecto.

- **App:** ENLACE_APP
- **Repositorio (código `app.py`, `procesamiento.py`, `requirements.txt` y datos):** https://github.com/santiagotoromesa-cell/aca1-aprendizaje-automatico

**Cómo se integra con este notebook:** la app aplica **las mismas reglas de limpieza** (archivo `procesamiento.py`) y entrena el Random Forest
con **los mismos hiperparámetros** que se eligieron aquí, que se exportan en `configuracion_modelo.json`. También muestra las métricas de
prueba y la gráfica de reales contra estimados a partir de `predicciones_prueba.csv`.

In [ ]:
# Archivos que usa la app de Streamlit (se suben al repositorio de GitHub junto con app.py)
configuracion_app = {
    "parametros_random_forest": parametros_finales,
    "variables_numericas": variables_numericas,
    "variables_texto": variables_texto,
    "numero_zonas": 4,
    "metricas_prueba": {fila["Modelo"]: {"MAE": int(fila["MAE (USD)"]), "RMSE": int(fila["RMSE (USD)"]), "R2": float(fila["R2"])}
                        for _, fila in comparacion.iterrows()},
    "barrios_entrenamiento": int(len(X_ent)),
    "barrios_prueba": int(len(X_pru))
}
with open("configuracion_modelo.json", "w", encoding="utf-8") as archivo:
    json.dump(configuracion_app, archivo, ensure_ascii=False, indent=2, default=str)

pd.DataFrame({"real": y_pru.values, "estimado": estimado_final.round(0)}).to_csv("predicciones_prueba.csv", index=False)
print("Archivos para la app guardados: configuracion_modelo.json y predicciones_prueba.csv")

# 8. Conclusiones

1. **Respuesta a la pregunta orientadora:** sí es posible. Con 9 variables sencillas, el Random Forest ajustado estima el valor de la vivienda
   típica de un barrio con un **error promedio de unos USD 27.000** y explica el **82 %** de las diferencias de precio entre barrios,
   frente a USD 79.300 de error si se respondiera siempre el promedio.
2. **La calidad de los datos pesó tanto como el algoritmo.** Rellenar los vacíos sin fuga de información, retirar los precios cortados y los
   barrios imposibles, y crear variables por hogar y de distancia dio al modelo información más clara. La distancia creada resultó la
   segunda variable más importante.
3. **El Random Forest supera a la regresión lineal** porque la relación entre las variables y el precio no es una línea recta. El ajuste de
   hiperparámetros añadió una mejora moderada (4 %) y permitió elegir una versión más liviana sin perder precisión.
4. **K-Means confirmó, sin conocer el precio,** que la ubicación (norte/sur) y el ingreso separan el mercado en zonas de valor distinto.

## Limitaciones
- **Datos de 1990:** el modelo estima precios **del momento del censo**; no sirve para dar precios actuales sin reentrenarlo con datos recientes.
- **Válido hasta USD 500.000:** los barrios por encima del tope se retiraron; para ellos el modelo no da una respuesta confiable.
- **Mayor error en los extremos** de precio (muy baratos y de USD 400.000 a 500.000).
- **Estima el barrio, no una casa:** el valor es el de la vivienda típica de la zona, no el de una vivienda específica.

# 9. Referencias

Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5–32. https://doi.org/10.1023/A:1010933404324

Pace, R. K., & Barry, R. (1997). *California housing prices* [Conjunto de datos]. Kaggle (publicado por C. Nugent).
https://www.kaggle.com/datasets/camnugent/california-housing-prices

Pace, R. K., & Barry, R. (1997). Sparse spatial autoregressions. *Statistics & Probability Letters, 33*(3), 291–297.
https://doi.org/10.1016/S0167-7152(96)00140-X

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V.,
Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, É. (2011). Scikit-learn: Machine learning in Python.
*Journal of Machine Learning Research, 12*, 2825–2830.

Rivera Flor, H. (2026). *Clase 3: Fundamentos del aprendizaje automático* [Diapositivas]. Corporación Unificada Nacional de Educación Superior.

Streamlit. (s. f.). *Streamlit documentation*. https://docs.streamlit.io